In [65]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import style
style.use('ggplot')

In [66]:
horas = ['9-10', '10-11', '11-12', '12-13', '13-14', '14-15', '15-16', '16-17', '17-18', '18-19']
lambdas = [50, 60, 58, 43, 35, 31, 44, 59, 65, 23]
h_h = [13.64, 13.64,13.64,13.64,13.64,13.64,13.64,13.64,13.64,13.64]
c_e = [27.29,27.29,27.29,27.29,27.29,27.29,27.29,27.29,27.29,27.29 ]

df = pd.DataFrame({
    'hora':horas,
    'lambda':lambdas,
    'Cs':h_h,
    'Cq':c_e
    
})
df

,hora,lambda,Cs,Cq
0,9-10,50,13.64,27.29
1,10-11,60,13.64,27.29
2,11-12,58,13.64,27.29
3,12-13,43,13.64,27.29
4,13-14,35,13.64,27.29
5,14-15,31,13.64,27.29
6,15-16,44,13.64,27.29
7,16-17,59,13.64,27.29
8,17-18,65,13.64,27.29
9,18-19,23,13.64,27.29


In [67]:
# Mezcla de operaciones
# probs = [0.33, 0.28, 0.25, 0.07, 0.03]
# tiempos = [4, 2, 2, 4, 8]

In [68]:
def calcular_mu(probs, tiempos):
    """
    Calcula la tasa de servicio μ (clientes por hora)
    a partir de probabilidades y tiempos promedio (en minutos).
    """
    
    # Probabilidad conocida
    p_conocida = sum(probs)
    
    # Tiempo promedio ponderado conocido
    t_prom_conocido = sum(p * t for p, t in zip(probs, tiempos))
    
    # Probabilidad restante
    p_rest = 1 - p_conocida
    
    # Tiempo promedio del resto (ajuste proporcional)
    t_resto = t_prom_conocido / p_conocida
    
    # Tiempo promedio total
    t_prom_min = t_prom_conocido + p_rest * t_resto
    
    # Convertimos a clientes por hora
    mu = 60 / t_prom_min
    
    return mu


In [69]:
import math

def metricas_mmc(lam, mu, c, Cs, Cq):

    Ct = Cs + Cq

    r = lam / mu
    rho = lam / (c * mu)

    if rho >= 1:
        return [None]*14

    suma = sum((r**k)/math.factorial(k) for k in range(c))
    termino_final = (r**c)/(math.factorial(c)*(1-rho))
    P0 = 1/(suma + termino_final)

    Lq = (P0 * (r**c) * rho) / (math.factorial(c) * (1-rho)**2)

    Wq_h = Lq / lam
    Ws_h = Wq_h + (1/mu)

    Ls = lam * Ws_h

    Wq_m = Wq_h * 60
    Ws_m = Ws_h * 60

    # =========================
    # COSTOS
    # =========================

    servidores_ocupados = Ls - Lq #good
    servidores_ociosos = c - servidores_ocupados

    CTS = Ct * servidores_ocupados #good

    CTS_idle = Cs * servidores_ociosos

    CTQ = lam * Cq * Wq_h

    CT = CTS + CTS_idle + CTQ

    return (
        rho,
        P0,
        Lq,
        Wq_h,
        Ws_h,
        Ls,
        Wq_m,
        Ws_m,
        servidores_ocupados,
        servidores_ociosos,
        CTS,
        CTS_idle,
        CTQ,
        CT
    )

In [70]:
# frecuencia de operaciones 9-10
probs09_10 = [0.36, 0.18, 0.23, 0.09, 0.02, 0.04]
tiempos09_10 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 10-11
probs10_11 = [0.36, 0.16, 0.21, 0.1, 0.04, 0.04]
tiempos10_11 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 11-12
probs11_12 = [0.36, 0.17, 0.20, 0.10, 0.04, 0.04]
tiempos11_12 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 12-13
probs12_13 = [0.36, 0.17, 0.20, 0.10, 0.05, 0.05]
tiempos12_13 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 13-14
probs13_14 = [0.35, 0.19, 0.27, 0.10, 0.04, 0.05]
tiempos13_14 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 14-15
probs14_15 = [0.37, 0.21, 0.22, 0.10, 0.04, 0.05]
tiempos14_15 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 15-16
probs15_16 = [0.39, 0.19, 0.19, 0.10, 0.03, 0.05]
tiempos15_16 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 16-17
probs16_17 = [0.36, 0.19, 0.17, 0.10, 0.04, 0.04]
tiempos16_17 = [4, 2, 2, 4, 8]

# frecuencia de operaciones 17-18
probs17_18 = [0.30, 0.19, 0.16, 0.13, 0.06, 0.03]
tiempos17_18 = [4, 2, 2, 4, 8]

probs18_17 = [0.30, 0.19, 0.16, 0.13, 0.06, 0.03]
tiempos17_18 = [4, 2, 2, 4, 8]

tipos = ["DEPOSITO", "COBRANZAS LOCALES OTRAS AGENCIAS", "RETIRO", "COBRANZAS", "DESEMBOLSOS", "APERTURA"]

df_operaciones = pd.DataFrame({
    "Operacion": tipos,
    "Tiempo_min": [2.5, 2.5, 2.5, 4.8, 20, 2.2],
    "9-10": probs09_10,
    "10-11": probs10_11,
    "11-12": probs11_12,
    "12-13": probs12_13,
    "13-14": probs13_14,
    "14-15": probs14_15,
    "15-16": probs15_16,
    "16-17": probs16_17,
    "17-18": probs17_18,
    "18-19":probs18_17
})

df_operaciones

,Operacion,Tiempo_min,9-10,10-11,11-12,12-13,13-14,14-15,15-16,16-17,17-18,18-19
0,DEPOSITO,2.5,0.36,0.36,0.36,0.36,0.35,0.37,0.39,0.36,0.30,0.30
1,COBRANZAS LOCALES OTRAS AGENCIAS,2.5,0.18,0.16,0.17,0.17,0.19,0.21,0.19,0.19,0.19,0.19
2,RETIRO,2.5,0.23,0.21,0.20,0.20,0.27,0.22,0.19,0.17,0.16,0.16
3,COBRANZAS,4.8,0.09,0.10,0.10,0.10,0.10,0.10,0.10,0.10,0.13,0.13
4,DESEMBOLSOS,20.0,0.02,0.04,0.04,0.05,0.04,0.04,0.03,0.04,0.06,0.06
5,APERTURA,2.2,0.04,0.04,0.04,0.05,0.05,0.05,0.05,0.04,0.03,0.03


In [71]:
# df["mu"] = df.apply(lambda row: calcular_mu(probs, tiempos), axis=1)
mu_por_hora = {}

tiempos = df_operaciones["Tiempo_min"].values

for hora in df_operaciones.columns[2:]:  # saltamos Operacion y Tiempo_min
    probs = df_operaciones[hora]
    mu_por_hora[hora] = calcular_mu(probs, tiempos)



df["mu"] = df["hora"].map(mu_por_hora)

df["s"] = 5


df[[
"rho","P0","Lq","Wq_h","Ws_h","Ls","Wq_m","Ws_m",
"ser_ocu","ser_idle",
"CTS","CTS_idle",
"CTQ","CT"
]] = df.apply(
    lambda row: metricas_mmc(
        row["lambda"],
        row["mu"],
        row["s"],
        row["Cq"],
        row["Cs"]
    ),
    axis=1,
    result_type="expand"
)

df


,hora,lambda,Cs,Cq,mu,s,rho,P0,Lq,Wq_h,Ws_h,Ls,Wq_m,Ws_m,ser_ocu,ser_idle,CTS,CTS_idle,CTQ,CT
0,9-10,50,13.64,27.29,19.402460,5,0.515399,0.073847,0.153493,0.003070,0.054610,2.730486,0.184192,3.276583,2.576993,2.423007,105.476313,66.123868,2.093645,173.693827
1,10-11,60,13.64,27.29,17.099906,5,0.701758,0.025614,0.895652,0.014928,0.073407,4.404443,0.895652,4.404443,3.508791,1.491209,143.614824,40.695088,12.216687,196.526599
2,11-12,58,13.64,27.29,17.099906,5,0.678366,0.029591,0.725931,0.012516,0.070996,4.117762,0.750963,4.259754,3.391832,1.608168,138.827663,43.886918,9.901698,192.616280
3,12-13,43,13.64,27.29,16.339678,5,0.526326,0.069682,0.171933,0.003998,0.065199,2.803564,0.239907,3.911950,2.631631,2.368369,107.712650,64.632795,2.345170,174.690615
4,13-14,35,13.64,27.29,17.569546,5,0.398417,0.135417,0.038974,0.001114,0.058030,2.031057,0.066813,3.481813,1.992083,3.007917,81.535971,82.086046,0.531606,164.153623
5,14-15,31,13.64,27.29,17.522124,5,0.353838,0.169811,0.020787,0.000671,0.057741,1.789978,0.040232,3.464474,1.769192,3.230808,72.413025,88.168753,0.283528,160.865306
6,15-16,44,13.64,27.29,18.298555,5,0.480912,0.088526,0.105841,0.002405,0.057055,2.510402,0.144328,3.423276,2.404561,2.595439,98.418698,70.829519,1.443669,170.691886
7,16-17,59,13.64,27.29,17.045455,5,0.692267,0.027175,0.822489,0.013940,0.072607,4.283822,0.836429,4.356429,3.461333,1.538667,141.672373,41.990213,11.218748,194.881335
8,17-18,65,13.64,27.29,14.850640,5,0.875383,0.006594,4.976050,0.076555,0.143892,9.352966,4.593277,8.633507,4.376916,0.623084,179.147160,17.003970,67.873325,264.024455
9,18-19,23,13.64,27.29,14.850640,5,0.309751,0.212110,0.010240,0.000445,0.067782,1.558995,0.026713,4.066943,1.548755,3.451245,63.390534,94.184482,0.139672,157.714687


In [76]:
def servidores_optimos(lam, mu, Cs, Cq, umbral_wq_m=5, rho_max=0.85, c_max=20):

    for c in range(1, c_max + 1):

        metrics = metricas_mmc(lam, mu, c, Cs, Cq)

        if metrics[0] is None:
            continue

        rho = metrics[0]
        Wq_m = metrics[6]

        if rho < rho_max and Wq_m <= umbral_wq_m:

            return (c,) + metrics

    return (None,) * 15


df[[
"s_o",
"rho_o",
"P0_o",
"Lq_o",
"Wq_h_o",
"Ws_h_o",
"Ls_o",
"Wq_m_o",
"Ws_m_o",
"ser_ocu_o",
"ser_idle_o",
"CTS_o",
"CTS_idle_o",
"CTQ_o",
"CT_o"
]] = df.apply(
    lambda row: servidores_optimos(
        row["lambda"],
        row["mu"],
        row["Cs"],   # Cs
        row["Cq"]    # Cq
    ),
    axis=1,
    result_type="expand"
)

df

,hora,lambda,Cs,Cq,mu,s,rho,P0,Lq,Wq_h,...,Wq_m_o,Ws_m_o,serv_ocup_o,serv_ocio_o,CTS_o,CTS_idle_o,CTQ_o,CT_o,ser_ocu_o,ser_idle_o
0,9-10,50,13.64,27.29,19.402460,5,0.515399,0.073847,0.153493,0.003070,...,0.752591,3.844982,2.576993,1.423007,105.476313,19.409819,17.115176,142.001308,2.576993,1.423007
1,10-11,60,13.64,27.29,17.099906,5,0.701758,0.025614,0.895652,0.014928,...,0.895652,4.404443,3.508791,1.491209,143.614824,20.340088,24.442330,188.397242,3.508791,1.491209
2,11-12,58,13.64,27.29,17.099906,5,0.678366,0.029591,0.725931,0.012516,...,3.954441,7.463232,3.391832,0.608168,138.827663,8.295418,104.319462,251.442544,3.391832,0.608168
3,12-13,43,13.64,27.29,16.339678,5,0.526326,0.069682,0.171933,0.003998,...,0.981396,4.653439,2.631631,1.368369,107.712650,18.664556,19.193974,145.571179,2.631631,1.368369
4,13-14,35,13.64,27.29,17.569546,5,0.398417,0.135417,0.038974,0.001114,...,1.493294,4.908294,1.992083,1.007917,81.535971,13.747983,23.771997,119.055951,1.992083,1.007917
5,14-15,31,13.64,27.29,17.522124,5,0.353838,0.169811,0.020787,0.000671,...,0.950608,4.374851,1.769192,1.230808,72.413025,16.788222,13.403416,102.604664,1.769192,1.230808
6,15-16,44,13.64,27.29,18.298555,5,0.480912,0.088526,0.105841,0.002405,...,3.577569,6.856517,2.404561,0.595439,98.418698,8.121782,71.596699,178.137179,2.404561,0.595439
7,16-17,59,13.64,27.29,17.045455,5,0.692267,0.027175,0.822489,0.013940,...,0.836429,4.356429,3.461333,1.538667,141.672373,20.987413,22.445722,185.105508,3.461333,1.538667
8,17-18,65,13.64,27.29,14.850640,5,0.875383,0.006594,4.976050,0.076555,...,0.958913,4.999143,4.376916,1.623084,179.147160,22.138870,28.349458,229.635488,4.376916,1.623084
9,18-19,23,13.64,27.29,14.850640,5,0.309751,0.212110,0.010240,0.000445,...,0.708678,4.748908,1.548755,1.451245,63.390534,19.794985,7.413600,90.599118,1.548755,1.451245


In [73]:
import simpy
import random
import numpy as np

def simular_agencia(lam, mu, ventanillas, tiempo_sim=480):

    esperas = []

    def cliente(env, ventanillas):

        llegada = env.now

        with ventanillas.request() as req:
            yield req

            espera = env.now - llegada
            esperas.append(espera)

            servicio = random.expovariate(mu)
            yield env.timeout(servicio)

    def generador(env, ventanillas):

        while True:
            yield env.timeout(random.expovariate(lam))
            env.process(cliente(env, ventanillas))

    env = simpy.Environment()
    recurso = simpy.Resource(env, capacity=ventanillas)

    env.process(generador(env, recurso))

    env.run(until=tiempo_sim)

    return {
        "clientes": len(esperas),
        "espera_promedio": np.mean(esperas),
        "p95": np.percentile(esperas,95)
    }

In [74]:
simulaciones = []

for _, row in df.iterrows():

    lam = row["lambda"]/60
    mu = row["mu"]/60

    for v in range(2,10):

        resultado = simular_agencia(lam, mu, v)

        simulaciones.append({
            "hora": row["hora"],
            "ventanillas": v,
            "espera_promedio": resultado["espera_promedio"],
            "espera_p95": resultado["p95"],
            "clientes": resultado["clientes"]
        })

df_sim = pd.DataFrame(simulaciones)

df_sim

,hora,ventanillas,espera_promedio,espera_p95,clientes
0,9-10,2,78.724981,141.876421,298
1,9-10,3,9.641365,21.507906,392
2,9-10,4,0.647943,4.454871,391
3,9-10,5,0.237367,1.927386,427
4,9-10,6,0.070876,0.378987,406
...,...,...,...,...,...
75,18-19,5,0.056871,0.052469,180
76,18-19,6,0.000000,0.000000,187
77,18-19,7,0.000000,0.000000,167
78,18-19,8,0.000000,0.000000,183


In [75]:
def sensibilidad_demanda(lam, mu, s):

    cambios = [-0.2,-0.1,0,0.1,0.2]

    resultados = []

    for c in cambios:

        lam_nuevo = lam * (1+c)

        rho, P0, Lq, Wq_h, Ws_h, Ls, Wq_m, Ws_m = metricas_mmc(lam_nuevo, mu, s)

        resultados.append({
            "variacion_demanda":c,
            "lambda":lam_nuevo,
            "rho":rho,
            "Wq_min":Wq_m
        })
    
    return pd.DataFrame(resultados)

In [77]:
escenarios = [0.8, 0.9, 1, 1.1, 1.2, 1.3]

resultados = []

for factor in escenarios:

    df_temp = df.copy()
    df_temp["lambda_sens"] = df_temp["lambda"] * factor

    df_temp[[
        "rho_s",
        "P0_s",
        "Lq_s",
        "Wq_h_s",
        "Ws_h_s",
        "Ls_s",
        "Wq_m_s",
        "Ws_m_s",
        "ser_ocu_s",
        "ser_idle_s",
        "CTS_s",
        "CTS_idle_s",
        "CTQ_s",
        "CT_s"
    ]] = df_temp.apply(
        lambda row: metricas_mmc(
            row["lambda_sens"],
            row["mu"],
            row["s"],
            row["Cs"],
            row["Cq"]
        ),
        axis=1,
        result_type="expand"
    )

    costo_total = df_temp["CT_s"].sum()

    resultados.append({
        "factor_demanda": factor,
        "costo_total": costo_total,
        "espera_promedio": df_temp["Wq_m_s"].mean(),
        "rho_promedio": df_temp["rho_s"].mean()
    })

df_sensibilidad = pd.DataFrame(resultados)

df_sensibilidad

,factor_demanda,costo_total,espera_promedio,rho_promedio
0,0.8,1332.619882,0.214413,0.442593
1,0.9,1458.612063,0.393953,0.497918
2,1.0,1653.073061,0.777851,0.553242
3,1.1,2304.210739,2.509841,0.608566
4,1.2,1658.277361,1.019748,0.620938
5,1.3,2057.680436,2.033960,0.672683
